# Evaluación del agente — Unidad 5

Se evalúa el agente sobre 4 casos midiendo:

- **Trayectoria:** herramientas esperadas vs. herramientas invocadas, leídas de los `tool_calls` de los mensajes del agente (no del texto final).
- **Resultado:** la decisión que devolvió la herramienta de reglas coincide con la esperada, calculada de forma determinista, y la respuesta final es coherente con esa decisión.

| Caso | Qué prueba | Herramientas esperadas |
|---|---|---|
| 1 | Saludo | ninguna |
| 2 | Pregunta que solo necesita las reglas | `evaluar_regla_zona` |
| 3 | Caso completo: imagen de la zona + reglas | `clasificar_zona` → `evaluar_regla_zona` |
| 4 | Caso completo con otra imagen, en el mismo `thread_id` del caso 3 y sin repetir zona ni hora (prueba también la memoria) | `clasificar_zona` → `evaluar_regla_zona` |

> Requiere haber ejecutado `entrenamiento_u4.ipynb` (modelo en `models/` e imágenes en `assets/ejemplos/`) y tener `GOOGLE_API_KEY` en `.env`.

In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
load_dotenv(RAIZ / ".env")

from src import reglas, tools
from src.agent import construir_agente, nuevo_thread_id, preguntar
from src.config import DIR_ASSETS, DIR_EJEMPLOS, MODELO_GEMINI
from src.vision import ClasificadorZona, leer_imagen_bgr

import os
print("GOOGLE_API_KEY cargada:", "sí" if os.getenv("GOOGLE_API_KEY") else "NO — revisen el archivo .env")
print("Modelo LLM:", MODELO_GEMINI)

In [ ]:
clasificador = ClasificadorZona()
tools.configurar_clasificador(clasificador)
agente = construir_agente()
print("Red neuronal cargada:", clasificador.nombre)
print("Agente construido con herramientas:", [h.name for h in tools.HERRAMIENTAS])

In [ ]:
# Imágenes de los casos 3 y 4: los ejemplos que el notebook de entrenamiento guardó en assets/ejemplos
def cargar_ejemplo(patron):
    rutas = sorted(DIR_EJEMPLOS.glob(patron))
    assert rutas, f"No hay assets/ejemplos/{patron}: ejecuten primero entrenamiento_u4.ipynb"
    imagen = leer_imagen_bgr(rutas[0])
    imagen_id = tools.registrar_imagen(imagen, etiqueta=rutas[0].name)
    prediccion = clasificador.predecir(imagen)
    print("Imagen:", rutas[0].name, "| ID:", imagen_id, "| salida directa de la red:", prediccion)
    return imagen_id, prediccion


id_caso3, pred_caso3 = cargar_ejemplo("ejemplo_ocupada_*.jpg")
id_caso4, pred_caso4 = cargar_ejemplo("ejemplo_vacia_*.jpg")

# Decisiones esperadas, calculadas sin LLM con las mismas reglas
esperada_2 = reglas.evaluar("caja", "15:40", "vacía", 8)["decision"]
esperada_3 = reglas.evaluar("bodega", "23:30", pred_caso3["estado"], 0, pred_caso3["confianza"])["decision"]
esperada_4 = reglas.evaluar("bodega", "23:30", pred_caso4["estado"], 0, pred_caso4["confianza"])["decision"]
print("Decisiones esperadas → caso 2:", esperada_2, "| caso 3:", esperada_3, "| caso 4:", esperada_4)

In [ ]:
HILO_MEMORIA = nuevo_thread_id()   # los casos 3 y 4 comparten conversación

casos = [
    {"caso": 1, "que_prueba": "Saludo, sin herramientas",
     "mensaje": "Hola, ¿qué puedes hacer por mí?",
     "thread_id": nuevo_thread_id(), "esperadas": [], "decision_esperada": None, "memoria": None},
    {"caso": 2, "que_prueba": "Solo reglas (@tool #2)",
     "mensaje": "Son las 15:40 y la caja lleva 8 minutos sin nadie. ¿Debo enviar aviso?",
     "thread_id": nuevo_thread_id(), "esperadas": ["evaluar_regla_zona"], "decision_esperada": esperada_2,
     "memoria": None},
    {"caso": 3, "que_prueba": "Completo: red neuronal + reglas",
     "mensaje": f"Revisa la bodega en esta imagen; son las 23:30. ¿Hay que avisar?\n(imagen adjunta: {id_caso3})",
     "thread_id": HILO_MEMORIA, "esperadas": ["clasificar_zona", "evaluar_regla_zona"],
     "decision_esperada": esperada_3, "memoria": None},
    {"caso": 4, "que_prueba": "Completo con otra imagen + memoria (mismo thread_id del caso 3)",
     "mensaje": f"Llegó otra imagen de la misma zona, a la misma hora. ¿Y ahora?\n(imagen adjunta: {id_caso4})",
     "thread_id": HILO_MEMORIA, "esperadas": ["clasificar_zona", "evaluar_regla_zona"],
     "decision_esperada": esperada_4, "memoria": {"zona": "bodega", "hora": "23:30"}},
]

PALABRAS_CLAVE = {
    "ALERTA": ["aviso", "avisar", "alerta"],
    "SIN ALERTA": ["no ", "no es necesario", "no hace falta", "sin alerta"],
    "VERIFICAR": ["revis", "verific", "confirm"],
}


def leer_decision(salidas):
    for salida in reversed(salidas):
        if "decision=" in salida:
            campos = dict(parte.split("=", 1) for parte in salida.split(" | ") if "=" in parte)
            return campos.get("decision")
    return None


def recordo_contexto(herramientas, esperado):
    # Memoria: en el caso 4 el usuario no repite zona ni hora; deben salir del turno anterior del mismo hilo
    for h in herramientas:
        if h["nombre"] == "evaluar_regla_zona":
            zona_ok = str(h["args"].get("zona", "")).strip().lower() == esperado["zona"]
            try:
                hora_ok = reglas.a_minutos(str(h["args"].get("hora", ""))) == reglas.a_minutos(esperado["hora"])
            except ValueError:
                hora_ok = False
            return zona_ok and hora_ok
    return False


def evaluar_caso(caso):
    r = preguntar(agente, caso["mensaje"], caso["thread_id"])
    usadas = [h["nombre"] for h in r["herramientas"]]          # trayectoria desde tool_calls
    trayectoria_ok = usadas == caso["esperadas"]
    decision = leer_decision(r["salidas_herramientas"])
    memoria_ok = None if caso["memoria"] is None else recordo_contexto(r["herramientas"], caso["memoria"])
    if caso["decision_esperada"] is None:
        resultado_ok = len(usadas) == 0 and len(r["respuesta"].strip()) > 0
    else:
        coherente = any(p in r["respuesta"].lower() for p in PALABRAS_CLAVE.get(caso["decision_esperada"], []))
        resultado_ok = decision == caso["decision_esperada"] and coherente and memoria_ok is not False
    return {**caso, "invocadas": usadas, "args": [h["args"] for h in r["herramientas"]],
            "salidas": r["salidas_herramientas"], "decision_obtenida": decision, "respuesta": r["respuesta"],
            "trayectoria_ok": trayectoria_ok, "memoria_ok": memoria_ok, "resultado_ok": resultado_ok,
            "tokens": r["tokens"]["total_tokens"], "mensajes_en_hilo": r["mensajes_en_hilo"]}


resultados = []
for caso in casos:
    ev = evaluar_caso(caso)
    resultados.append(ev)
    print(f"=== Caso {ev['caso']} · {ev['que_prueba']} ===")
    print("Mensaje:", ev["mensaje"].replace("\n", " "))
    print("Esperadas:", ev["esperadas"], "| Invocadas:", ev["invocadas"])
    for args, salida in zip(ev["args"], ev["salidas"]):
        print("  args:", args)
        print("  salida:", salida)
    print("Decisión esperada:", ev["decision_esperada"], "| obtenida:", ev["decision_obtenida"])
    if ev["memoria_ok"] is not None:
        print("¿Recordó zona y hora del turno anterior?:", "sí" if ev["memoria_ok"] else "NO")
    print("Respuesta:", ev["respuesta"])
    print(f"[{'OK' if ev['trayectoria_ok'] else 'FALLÓ'} trayectoria] [{'OK' if ev['resultado_ok'] else 'FALLÓ'} resultado]"
          f" tokens del turno={ev['tokens']} | mensajes acumulados en el hilo={ev['mensajes_en_hilo']}\n")
    time.sleep(5)   # pausa para no superar el límite de solicitudes por minuto del plan gratuito

In [ ]:
matriz = pd.DataFrame([{
    "caso": r["caso"], "qué prueba": r["que_prueba"], "esperadas": " → ".join(r["esperadas"]) or "ninguna",
    "invocadas": " → ".join(r["invocadas"]) or "ninguna", "trayectoria": "OK" if r["trayectoria_ok"] else "FALLÓ",
    "decisión esperada": r["decision_esperada"] or "—", "decisión obtenida": r["decision_obtenida"] or "—",
    "memoria": "—" if r["memoria_ok"] is None else ("OK" if r["memoria_ok"] else "FALLÓ"),
    "resultado": "OK" if r["resultado_ok"] else "FALLÓ", "tokens": r["tokens"],
} for r in resultados])
print(matriz.to_string(index=False))

exactitud_trayectoria = sum(r["trayectoria_ok"] for r in resultados) / len(resultados)
exactitud_resultado = sum(r["resultado_ok"] for r in resultados) / len(resultados)
print(f"\nExactitud de trayectoria: {exactitud_trayectoria:.0%}")
print(f"Exactitud de resultado:   {exactitud_resultado:.0%}")
print(f"Tokens totales de la evaluación: {sum(r['tokens'] for r in resultados)}")

matriz.to_csv(DIR_ASSETS / "evaluacion_agente.csv", index=False)
print("Matriz guardada en assets/evaluacion_agente.csv")